# Telco Churn - Production Aligned Notebook

This notebook follows the same logic used in `src/pipelines/train_pipeline.py` so your experiments are consistent with production.

In [ ]:
from pathlib import Path
import pandas as pd

from src.data.dataset import load_dataset, prepare_training_data
from src.features.preprocessing import build_preprocessor
from src.training.models import build_model_pipeline, get_model_candidates
from src.evaluation.metrics import tune_threshold, compute_classification_metrics
from sklearn.model_selection import train_test_split


In [ ]:
DATA_PATH = Path(r"C:/Users/soumy/OneDrive/Desktop/churnProject/churn-platform/ibm/Telco_customer_churn.xlsx")
raw_df = load_dataset(str(DATA_PATH))
features_raw, target, target_column = prepare_training_data(raw_df)
print('Rows:', len(features_raw), 'Target:', target_column)
features_raw.head()

In [ ]:
# Keep the model input contract aligned with Node -> FastAPI payload
feature_columns = [
    'Tenure in Months',
    'Monthly Charge',
    'Contract',
    'Internet Service',
    'Tech Support'
]

alias_map = {
    'Tenure in Months': 'tenure_months',
    'Monthly Charge': 'monthly_charges',
    'Contract': 'contract',
    'Internet Service': 'internet_service',
    'Tech Support': 'tech_support',
}

X = features_raw[feature_columns].rename(columns=alias_map)
y = target.copy()
X.head()

In [ ]:
x_train_val, x_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)
print(x_train.shape, x_val.shape, x_test.shape)

In [ ]:
candidates = get_model_candidates(random_state=42)
results = []

for name, model in candidates.items():
    preprocessor, _, _ = build_preprocessor(X)
    pipe = build_model_pipeline(preprocessor, model)
    pipe.fit(x_train, y_train)
    val_prob = pipe.predict_proba(x_val)[:, 1]
    threshold, threshold_metrics = tune_threshold(y_val, val_prob, min_precision=0.0)
    metrics = compute_classification_metrics(y_val, val_prob, threshold)
    results.append((name, threshold, metrics, threshold_metrics))

for name, threshold, metrics, threshold_metrics in results:
    print(name, 'threshold=', round(threshold, 4), 'f1=', round(metrics['f1'], 4), 'pr_auc=', round(metrics['pr_auc'], 4))

## Promotion to Production

After experimentation, use the CLI pipeline to produce and copy artifacts:

```bash
python -m src.pipelines.train_pipeline --input-path "C:/Users/soumy/OneDrive/Desktop/churnProject/churn-platform/ibm/Telco_customer_churn.xlsx" --export-to-ml-api
```